📚 **Importación de libreías**

In [1]:
from pathlib import Path
import sys
#Pandas: Para manipulación y análisis de datos tabulares.
import pandas as pd
#NumPy: Para operaciones matemáticas y manejo de arrays.
import numpy as np
#Scikit-learn: Para preprocesamiento de datos, selección de características y transformación.
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
# Feature-engine: Para técnicas avanzadas de ingeniería de características.
from feature_engine.imputation import MeanMedianImputer
from feature_engine.encoding import OneHotEncoder

💾 **Carga de datos**

In [2]:
# Definir rutas de datos
data_path = Path("../../data")
bronze_path = data_path / "Bronze"

# Verificar que la carpeta existe
if not bronze_path.exists():
    print(f"❌ No se encontró la carpeta: {bronze_path}")
    sys.exit(1)

# Cargar el dataset principal
dataset_file = bronze_path / "Dataset.csv"

if dataset_file.exists():
    print(f"📂 Cargando datos desde: {dataset_file}")
    df = pd.read_csv(dataset_file)

    # Información básica del dataset
    print(f"✅ Dataset cargado exitosamente")
    print(f"📊 Forma del dataset: {df.shape}")
    print(f"📋 Columnas: {list(df.columns)}")
    print(f"💾 Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


📂 Cargando datos desde: ../../data/Bronze/Dataset.csv
✅ Dataset cargado exitosamente
📊 Forma del dataset: (1744667, 27)
📋 Columnas: ['CreditoID', 'PersonaID', 'DepartamentoResidencia', 'DepartamentoMayorFrecuenciaCompra', 'AlmacenMayorFrecuenciaPago', 'ValorPagosUltimosMes', 'AlmacenCredito', 'RiesgoAlmacen', 'DepartamentoCredito', 'FechaCredito', 'ValorCredito', 'CupoTotal', 'CupoDisponibleTotal', 'storeIdEventoA', 'FechaEventoA', 'EventoA', 'FechaEventoB', 'LocalizacionEventoB', 'LocalizacionComercioCredito', 'StatusComercioCredito', 'FrecuenciaCreditosSemana', 'CantidadCreditosUltimaSemana', 'ValorAtipicoCliente', 'ValorAtipicoComercio', 'TipoAlmacenCredito', 'TipoCliente', 'Atipico']
💾 Memoria utilizada: 1540.94 MB


**1. Limpieza de datos**
En esta etapa inicial, se realizaron tareas esenciales para asegurar la calidad y coherencia de los datos:

- Eliminación de duplicados
- Tratamiento de valores faltantes
- Atípicos

In [3]:
# Código de conversión
# Crear una copia para trabajar
df_converted = df.copy()

# 🆔 VARIABLES CATEGÓRICAS NOMINALES (IDs)
categorical_ids = [
    "CreditoID",
    "PersonaID",
    "AlmacenMayorFrecuenciaPago",
    "AlmacenCredito",
    "storeIdEventoA",
]

for col in categorical_ids:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("string")
        print(f"   ✅ {col}: Convertido a string (ID categórico)")

# 🏷️ VARIABLES CATEGÓRICAS NOMINALES (Códigos/Regiones)
categorical_nominal = [
    "DepartamentoResidencia",
    "DepartamentoMayorFrecuenciaCompra",
    "DepartamentoCredito",
    "TipoAlmacenCredito",
]

for col in categorical_nominal:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("category")
        print(f"   ✅ {col}: Convertido a category (nominal)")

# 📊 VARIABLES CATEGÓRICAS ORDINALES
categorical_ordinal = ["RiesgoAlmacen", "StatusComercioCredito", "TipoCliente"]

for col in categorical_ordinal:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("category")
        print(f"   ✅ {col}: Convertido a category (ordinal)")

# 📅 VARIABLES DE FECHA
date_columns = ["FechaCredito", "FechaEventoA", "FechaEventoB"]

for col in date_columns:
    if col in df_converted.columns:
        try:
            # Intentar múltiples formatos de fecha
            df_converted[col] = pd.to_datetime(
                df_converted[col], errors="coerce", infer_datetime_format=True
            )
            print(f"   ✅ {col}: Convertido a datetime")
        except:
            print(f"   ❌ {col}: Error al convertir a datetime")

# 💰 VARIABLES NUMÉRICAS CONTINUAS
numeric_continuous = [
    "ValorPagosUltimosMes",
    "ValorCredito",
    "CupoTotal",
    "CupoDisponibleTotal",
    "ValorAtipicoCliente",
    "ValorAtipicoComercio",
]

for col in numeric_continuous:
    if col in df_converted.columns:
        try:
            df_converted[col] = pd.to_numeric(df_converted[col], errors="coerce")
            print(f"   ✅ {col}: Convertido a numeric (continua)")
        except:
            print(f"   ❌ {col}: Error al convertir a numeric")

# 🔢 VARIABLES NUMÉRICAS DISCRETAS (Conteos)
numeric_discrete = ["FrecuenciaCreditosSemana", "CantidadCreditosUltimaSemana"]

for col in numeric_discrete:
    if col in df_converted.columns:
        try:
            df_converted[col] = pd.to_numeric(
                df_converted[col], errors="coerce"
            ).astype("Int64")
            print(f"   ✅ {col}: Convertido a Int64 (discreta)")
        except:
            print(f"   ❌ {col}: Error al convertir a Int64")

# ✅ VARIABLES BINARIAS
# EventoA: 'SI' vs NaN -> 1/0
if "EventoA" in df_converted.columns:
    df_converted["EventoA"] = (df_converted["EventoA"] == "SI").astype("Int64")
    print(f"   ✅ EventoA: Convertido a binario (1=SI, 0=NaN)")

# Atipico: Variable objetivo binaria
if "Atipico" in df_converted.columns:
    df_converted["Atipico"] = df_converted["Atipico"].astype("Int64")
    print(f"   ✅ Atipico: Convertido a Int64 (target binario)")

# 🗺️ VARIABLES GEOESPACIALES
geo_columns = ["LocalizacionEventoB", "LocalizacionComercioCredito"]

for col in geo_columns:
    if col in df_converted.columns:
        print(
            f"   📍 {col}: Mantenido como string (requiere procesamiento geoespacial)"
        )

print(f"\\n📊 RESUMEN DE CONVERSIONES APLICADAS")
print("=" * 70)

# Comparar tipos antes y después
changes_made = []
for col in df.columns:
    if col in df_converted.columns:
        old_type = str(df[col].dtype)
        new_type = str(df_converted[col].dtype)
        if old_type != new_type:
            changes_made.append(
                {"Columna": col, "Tipo_Original": old_type, "Tipo_Nuevo": new_type}
            )

if changes_made:
    changes_df = pd.DataFrame(changes_made)
    display(changes_df)
else:
    print("No se realizaron cambios de tipo de datos")

# Actualizar el dataframe principal
print(f"\\n✅ Dataset actualizado con nuevos tipos de datos")


   ✅ CreditoID: Convertido a string (ID categórico)
   ✅ PersonaID: Convertido a string (ID categórico)
   ✅ AlmacenMayorFrecuenciaPago: Convertido a string (ID categórico)
   ✅ AlmacenCredito: Convertido a string (ID categórico)
   ✅ storeIdEventoA: Convertido a string (ID categórico)
   ✅ DepartamentoResidencia: Convertido a category (nominal)
   ✅ DepartamentoMayorFrecuenciaCompra: Convertido a category (nominal)
   ✅ DepartamentoCredito: Convertido a category (nominal)
   ✅ TipoAlmacenCredito: Convertido a category (nominal)
   ✅ RiesgoAlmacen: Convertido a category (ordinal)
   ✅ StatusComercioCredito: Convertido a category (ordinal)
   ✅ TipoCliente: Convertido a category (ordinal)


/tmp/ipykernel_5445/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col] = pd.to_datetime(
/tmp/ipykernel_5445/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col] = pd.to_datetime(
/tmp/ipykernel_5445/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col] =

   ✅ FechaCredito: Convertido a datetime
   ✅ FechaEventoA: Convertido a datetime
   ✅ FechaEventoB: Convertido a datetime
   ✅ ValorPagosUltimosMes: Convertido a numeric (continua)
   ✅ ValorCredito: Convertido a numeric (continua)
   ✅ CupoTotal: Convertido a numeric (continua)
   ✅ CupoDisponibleTotal: Convertido a numeric (continua)
   ✅ ValorAtipicoCliente: Convertido a numeric (continua)
   ✅ ValorAtipicoComercio: Convertido a numeric (continua)
   ✅ FrecuenciaCreditosSemana: Convertido a Int64 (discreta)
   ✅ CantidadCreditosUltimaSemana: Convertido a Int64 (discreta)
   ✅ EventoA: Convertido a binario (1=SI, 0=NaN)
   ✅ Atipico: Convertido a Int64 (target binario)
   📍 LocalizacionEventoB: Mantenido como string (requiere procesamiento geoespacial)
   📍 LocalizacionComercioCredito: Mantenido como string (requiere procesamiento geoespacial)
\n📊 RESUMEN DE CONVERSIONES APLICADAS


,Columna,Tipo_Original,Tipo_Nuevo
0,CreditoID,object,string
1,PersonaID,object,string
2,DepartamentoResidencia,int64,category
3,DepartamentoMayorFrecuenciaCompra,int64,category
4,AlmacenMayorFrecuenciaPago,object,string
5,AlmacenCredito,object,string
6,RiesgoAlmacen,int64,category
7,DepartamentoCredito,int64,category
8,FechaCredito,object,"datetime64[ns, UTC]"
9,storeIdEventoA,object,string


\n✅ Dataset actualizado con nuevos tipos de datos


In [ ]:
# Verificar y eliminar duplicados por CreditoID
print("🔍 Análisis de duplicados por CreditoID")
print("=" * 50)

# Contar registros duplicados por CreditoID
duplicados = df_converted['CreditoID'].duplicated().sum()
print(f"📊 Registros duplicados encontrados: {duplicados}")
print(f"📊 Total de registros: {len(df_converted)}")

if duplicados > 0:
    # Mostrar algunos ejemplos de CreditoID duplicados
    creditos_duplicados = df_converted[df_converted['CreditoID'].duplicated(keep=False)]['CreditoID'].value_counts().head()
    print(f"\n🔍 Ejemplos de CreditoID duplicados:")
    print(creditos_duplicados)
    
    # Eliminar duplicados manteniendo el primer registro
    df_converted = df_converted.drop_duplicates(subset=['CreditoID'], keep='first')
    print(f"\n✅ Duplicados eliminados. Registros restantes: {len(df_converted)}")
else:
    print("✅ No se encontraron duplicados por CreditoID")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1744667 entries, 0 to 1744666
Data columns (total 27 columns):
 #   Column                             Dtype              
---  ------                             -----              
 0   CreditoID                          string             
 1   PersonaID                          string             
 2   DepartamentoResidencia             category           
 3   DepartamentoMayorFrecuenciaCompra  category           
 4   AlmacenMayorFrecuenciaPago         string             
 5   ValorPagosUltimosMes               float64            
 6   AlmacenCredito                     string             
 7   RiesgoAlmacen                      category           
 8   DepartamentoCredito                category           
 9   FechaCredito                       datetime64[ns, UTC]
 10  ValorCredito                       float64            
 11  CupoTotal                          float64            
 12  CupoDisponibleTotal                float64